# Optuna hyperparameter optimization

### Import libraries and set configs

In [1]:
import ast
import json
import pandas as pd

import optuna


class CFG:
    n_trials = 10
    # maximum number of simultaneously opened trades for backtest metric
    max_num_simult_trades = 100
    # significance level, that is used to conduct a t-test between 2 models
    optimize_alpha = 0.2
    n_repeats = 1
    n_folds = 8
    min_precision = 0.5
    TP = 0.05
    SL = 0.05
    slippage = 0.002

/home/alex/Repos/sigbot/.venv/lib/python3.12/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Load the train data

In [2]:
train_df = pd.read_pickle("data/train_df.pkl")

# all data for the last 90 days are test
test_date = train_df["time"].max() - pd.to_timedelta(90, unit="D")

profitable_hours_df = pd.read_csv("data/profitable_hours.csv")
latest = profitable_hours_df.iloc[-1]
buy_hours = ast.literal_eval(latest["profitable_buy_hours"])
sell_hours = ast.literal_eval(latest["profitable_sell_hours"])

buy_mask = (train_df["ttype"] == "buy") & (train_df["time"].dt.hour.isin(buy_hours))
sell_mask = (train_df["ttype"] == "sell") & (train_df["time"].dt.hour.isin(sell_hours))
train_df = train_df[buy_mask | sell_mask].reset_index(drop=True)

fi = pd.read_csv("model/feature_importance.csv")

### Optimize

In [ ]:
from utils.optimization_utils import make_objective

with open("model/bybit_tickers.json", "r") as f:
    bybit_tickers = json.load(f)

df_optuna_more_info = pd.DataFrame(columns=["result", "backtest_result", "oof_conf_score",
                                            "profit_objects", "oof_conf_obj_num", "scores"])
df_optuna_more_info.to_csv("model/optuna/optuna_lgbm_info.csv", index=False)

objective = make_objective(
    train_df, 
    test_date, 
    fi, 
    bybit_tickers, 
    TP=CFG.TP - CFG.slippage,
    SL=CFG.SL + CFG.slippage,
    n_folds=CFG.n_folds, 
    optimize_alpha=CFG.optimize_alpha, 
    min_precision=CFG.min_precision,
)
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=CFG.n_trials)

print("Number of finished trials: {}".format(len(study.trials)))

print("Best trial:")
trial = study.best_trial

print("  Value: {}".format(trial.value))

print("  Params: ")
for key, value in trial.params.items():
    print("    {}: {}".format(key, value))

df_optuna = study.trials_dataframe()
df_optuna = df_optuna.sort_values("value", ascending=False)
# df_optuna.to_csv("optuna/optuna_lgbm.csv", index=False)

display(df_optuna.head(10))

/home/alex/Repos/sigbot/.venv/lib/python3.12/site-packages/hyperopt/atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


OSError: Cannot save file into a non-existent directory: 'optuna'